# 2D случай

In [7]:
import matplotlib.pyplot as plt
import numpy as np

In [9]:
class Robot:
    def __init__(
            self,
            link_lengths=[50, 40, 30, 30],
            theta_array=[np.pi/2, 0, 0, 0]
        ):
        self.links = np.asarray(link_lengths)
        self.set_pose(theta_array)
        
    def forward_kinematics(self, theta_array, change_robot_conf=False) -> np.ndarray:
        if change_robot_conf:
            self.set_pose(theta_array)
        f_fk = np.zeros((self.links.size, 2))
        phi = 0
        J_m = np.zeros(2)
        for i in range(self.links.size):
            phi += theta_array[i]
            J_m += self.links[i] * np.array([np.cos(phi), np.sin(phi)])
            f_fk[i] = J_m
        return f_fk

    def column_jacobi(self, joint_num, col_num) -> np.array:
        total_sum = np.zeros(2)
        phi = np.sum(self._Thetas[:col_num])
        for i in range(col_num, joint_num+1):
            phi += self._Thetas[i]
            total_sum += self.links[i] * np.array([-np.sin(phi), np.cos(phi)])
        return total_sum
    
    def jacobi(self, joint_num) -> np.ndarray:
        jacobi_matrix = np.zeros((2, self.links.size))
        for i in range(self.links.size):
            jacobi_matrix[:, i] = self.column_jacobi(joint_num, i)
        return jacobi_matrix

    def set_pose(self, theta_array) -> None:
        self._Thetas = np.asarray(theta_array)
        assert self.links.size == self._Thetas.size, "Size of theta_array must be equal to link_lengths's size!"
    
    def get_pose(self) -> np.array:
        return self._Thetas

In [ ]:
def loss_and_grad(robot, theta, target_indices, target_points):
    joints = robot.forward_kinematics(theta, change_robot_conf=True)

    loss = 0.0
    grad = np.zeros_like(theta, dtype=float)

    for joint_idx, target in zip(target_indices, target_points):
        residual = joints[joint_idx] - target
        Jacobi = robot.jacobi(joint_idx)

        loss += residual.T @ residual
        grad += 2 * Jacobi.T @ residual

    return loss, grad

def solve_regression(
    robot,
    target_indices,
    target_points,
    theta_init=None,
    lr=1e-5,
    max_iter=10000,
    tol=1e-6,
):
    if theta_init is None:
        theta = robot.get_pose().copy()
    else:
        theta = np.asarray(theta_init, dtype=float).copy()

    history = []

    for step in range(max_iter):
        loss, grad = loss_and_grad(robot, theta, target_indices, target_points)
        history.append(loss)

        if np.linalg.norm(grad) < tol:
            break

        theta -= lr * grad
        theta = (theta + np.pi) % (2 * np.pi) - np.pi

    robot.set_pose(theta)
    return theta, history

3